In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [ ]:
data=pd.read_csv("Churn_Modelling.csv")
data

In [ ]:
# pre-process the data
## drop irrelevant columns
data=data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data

In [ ]:
## apply encoding on categorical variable for example geography, gender
label_encoder_gender=LabelEncoder()
data["Gender"]=label_encoder_gender.fit_transform(data["Gender"])
data

In [ ]:
one_hot_encoder_geography=OneHotEncoder(handle_unknown="ignore")
geo_encoded=one_hot_encoder_geography.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=one_hot_encoder_geography.get_feature_names_out(["Geography"]))
geo_encoded_df

In [ ]:
data=pd.concat([data.drop("Geography",axis=1),geo_encoded_df],axis=1)
data.head()

In [ ]:
# split the features and target
X = data.drop("EstimatedSalary",axis=1)
y = data["EstimatedSalary"]

In [ ]:
## split the data into training and testing data
X_train,X_test,Y_train,Y_test=train_test_split(X,y, test_size=0.2, random_state=42)

In [ ]:
# scale these feature
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [ ]:
## save the encoder and scaler
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump(one_hot_encoder_geography, file)

with open("scaler.pkl","wb") as file:
    pickle.dump(scaler, file)    

## ANN Regression Problem Statement

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# Build the model
model=Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)), ## HL1 connected to input layer
    Dense(32,activation="relu"), ## HL2
    Dense(1)
])

In [ ]:
## compile the model
model.compile(optimizer="adam", loss="mean_absolute_error", metrics=["mae"])

model.summary()

In [ ]:
## Setup the tenserboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callabcks=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
ealry_stopping_callback=EarlyStopping(monitor="val_loss", patience=14, restore_best_weights=True) ## patience till 5 epoch before decide gradient decendent

In [ ]:
## training the model
history=model.fit(
    X_train,Y_train,
    validation_data=(X_test,Y_test),
    epochs=100,
    callbacks=[tensorflow_callabcks, ealry_stopping_callback]
)

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir regressionlogs/fit

In [ ]:
## evaluate the mode
test_loss, test_mae = model.evaluate(X_test, Y_test)
print(f'test me: {test_mae}')

In [ ]:
model.save("regression_model.h5")